## Step 1: find trailer video IDs for every movie and store them in HBase

In [1]:
# Install the needed Python packages
%pip -q install happybase thriftpy2 google-api-python-client vaderSentiment tqdm pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports + config
import os
import re
import time
import hashlib
from datetime import datetime

import happybase
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [3]:
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
print("Key loaded:", bool(YOUTUBE_API_KEY))
if not YOUTUBE_API_KEY:
    raise ValueError("Missing YOUTUBE_API_KEY. Set it as an env var or paste it in directly.")

Key loaded: True


In [4]:
# alte Verbindung schließen
try:
    connection.close()
except Exception:
    pass

# neue Verbindung
connection = happybase.Connection(host="hbase", port=9090)
connection.open()

print(connection.tables())

[]


In [5]:
# === CONFIGURATION ===

MOVIES_TABLE = "movie_success_rate_raw"
TRAILERS_TABLE = "youtube_trailers"   
KINOCHECK_VIDEOS_TABLE = "kinocheck_videos"

TITLE_COL = b"cf:title"
YEAR_COL  = b"cf:year"

print("Using movies table:", MOVIES_TABLE)
print("Title column:", TITLE_COL)
print("Year column:", YEAR_COL)


Using movies table: movie_success_rate_raw
Title column: b'cf:title'
Year column: b'cf:year'


In [6]:
existing = [t.decode() for t in connection.tables()]

if TRAILERS_TABLE not in existing:
    connection.create_table(
        TRAILERS_TABLE,
        {
            "yt": dict(),     # video info
            "meta": dict(),   # search details / status
        }
    )
    print("Created table:", TRAILERS_TABLE)
else:
    print("Table already exists:", TRAILERS_TABLE)

# HBase Tabelle für KinoCheck-Videos anlegen

existing_tables = [t.decode() for t in connection.tables()]
if KINOCHECK_VIDEOS_TABLE not in existing_tables:
    connection.create_table(
        KINOCHECK_VIDEOS_TABLE,
        {
            "yt": dict(),   # YouTube-spezifische Infos
        }
    )
    print("Created table:", KINOCHECK_VIDEOS_TABLE)
else:
    print("Table already exists:", KINOCHECK_VIDEOS_TABLE)

kinocheck_videos = connection.table(KINOCHECK_VIDEOS_TABLE)
movies = connection.table(MOVIES_TABLE)
trailers = connection.table(TRAILERS_TABLE)

Created table: youtube_trailers
Created table: kinocheck_videos


In [7]:
def clean_title(s: str) -> str:
    if s is None:
        return ""
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9äöüß çéèáàíìóòúù'-]", " ", s)  # keep some common chars
    s = re.sub(r"\s+", " ", s).strip()
    return s

def make_movie_key(title: str, year: str) -> str:
    # stable key even if the dataset has no numeric id
    base = f"{clean_title(title)}|{str(year).strip()}"
    return hashlib.md5(base.encode("utf-8")).hexdigest()

In [8]:
movies = connection.table(MOVIES_TABLE)

# pull 1 row to inspect its columns
one = next(movies.scan(limit=1))
one

IOError: IOError(message=b'org.apache.hadoop.hbase.TableNotFoundException: movie_success_rate_raw\n\tat org.apache.hadoop.hbase.client.ConnectionImplementation.locateRegionInMeta(ConnectionImplementation.java:860)\n\tat org.apache.hadoop.hbase.client.ConnectionImplementation.locateRegion(ConnectionImplementation.java:755)\n\tat org.apache.hadoop.hbase.client.RpcRetryingCallerWithReadReplicas.getRegionLocations(RpcRetryingCallerWithReadReplicas.java:326)\n\tat org.apache.hadoop.hbase.client.ScannerCallableWithReplicas.call(ScannerCallableWithReplicas.java:153)\n\tat org.apache.hadoop.hbase.client.ScannerCallableWithReplicas.call(ScannerCallableWithReplicas.java:58)\n\tat org.apache.hadoop.hbase.client.RpcRetryingCallerImpl.callWithoutRetries(RpcRetryingCallerImpl.java:192)\n\tat org.apache.hadoop.hbase.client.ClientScanner.call(ClientScanner.java:266)\n\tat org.apache.hadoop.hbase.client.ClientScanner.loadCache(ClientScanner.java:434)\n\tat org.apache.hadoop.hbase.client.ClientScanner.n

In [9]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime
import time

youtube = build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)

In [10]:
# KinoCheck Channel ID über die YouTube API holen

KINOCHECK_CHANNEL_NAME = "KinoCheck"

channel_search_response = youtube.search().list(
    part="snippet",
    q=KINOCHECK_CHANNEL_NAME,
    type="channel",
    maxResults=1
).execute()

KINOCHECK_CHANNEL_ID = channel_search_response["items"][0]["id"]["channelId"]
print("Gefundene KinoCheck Channel ID:", KINOCHECK_CHANNEL_ID)

# Erwartete ID (Ich habe die ID schon über Postman geholt und gleiche nur noch ab)
KINOCHECK_CHANNEL_ID_KNOWN = "UCOL10n-as9dXO2qtjjFUQbQ"
print("Erwartete Channel ID:", KINOCHECK_CHANNEL_ID_KNOWN)

assert KINOCHECK_CHANNEL_ID == KINOCHECK_CHANNEL_ID_KNOWN, "Channel ID stimmt nicht mit der erwarteten überein!"


Gefundene KinoCheck Channel ID: UCOL10n-as9dXO2qtjjFUQbQ
Erwartete Channel ID: UCOL10n-as9dXO2qtjjFUQbQ


In [11]:
# KinoCheck Uploads Playlist ID über die YouTube API holen
# Über die Upload ID bekommt man mit einem Aufruf 50 Video IDs mit einem Aufruf

channels_response = youtube.channels().list(
    part="contentDetails",
    id=KINOCHECK_CHANNEL_ID
).execute()

UPLOADS_PLAYLIST_ID = channels_response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print("Gefundene Uploads Playlist ID:", UPLOADS_PLAYLIST_ID)

# Erwartete Uploads Playlist ID (auch bereits von Postman bekannt)
UPLOADS_PLAYLIST_ID_KNOWN = "UUOL10n-as9dXO2qtjjFUQbQ"
print("Erwartete Uploads Playlist ID:", UPLOADS_PLAYLIST_ID_KNOWN)

assert UPLOADS_PLAYLIST_ID == UPLOADS_PLAYLIST_ID_KNOWN, "Uploads Playlist ID stimmt nicht mit der erwarteten überein!"


Gefundene Uploads Playlist ID: UUOL10n-as9dXO2qtjjFUQbQ
Erwartete Uploads Playlist ID: UUOL10n-as9dXO2qtjjFUQbQ


In [12]:
# 250 Videos aus der KinoCheck Uploads-Playlist holen und in HBase speichern

MAX_VIDEOS = 250

saved = 0
next_page_token = None

while saved < MAX_VIDEOS:
    request = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=UPLOADS_PLAYLIST_ID,
        maxResults=50,
        pageToken=next_page_token
    )
    response = request.execute()

    items = response.get("items", [])
    if not items:
        print("Keine weiteren Videos gefunden.")
        break

    for item in items:
        video_id = item["contentDetails"]["videoId"]
        title = item["snippet"]["title"]

        rowkey = video_id.encode("utf-8")

        # Video ID + Titel speichern
        kinocheck_videos.put(rowkey, {
            b"yt:video_id": rowkey,
            b"yt:title": title.encode("utf-8", errors="ignore"),
        })

        saved += 1
        if saved >= MAX_VIDEOS:
            break

    next_page_token = response.get("nextPageToken")
    if not next_page_token:
        print("Playlist hat weniger als", MAX_VIDEOS, "Videos.")
        break

print("Anzahl gespeicherter Videos:", saved)



Anzahl gespeicherter Videos: 250


In [13]:
# Stichprobe: prüfen ob Videos speichern geklappt hat

for rk, data in kinocheck_videos.scan(limit=10):
    video_id = rk.decode("utf-8", errors="ignore")
    title = data.get(b"yt:title", b"").decode("utf-8", errors="ignore")
    print(video_id, "->", title)


-Di0wZ4PekI -> Hexenverbrennung ohne Gnade! - DER MEDICUS 2 Clip & Trailer German Deutsch (2025)
-JP9oWPK-u8 -> MOTHER'S BABY Trailer German Deutsch (2025) Exklusiv
-KEQV62AJm0 -> Die besten COMEDY Filme 2025 & 2026 (Trailer German Deutsch)
-KXQttVDPyk -> DER ASTRONAUT Trailer 2 German Deutsch (2026) Ryan Gosling, Sandra Hüller
-bNiFerldp4 -> SPRINGSTEEN: Deliver Me From Nowhere Trailer 2 German Deutsch (2025)
0ITMqStggGo -> Martin Freeman ist der Hüter der Zeit! - MOMO Clip & Trailer German Deutsch (2025)
0cxn0iqHcDM -> LEBENSGEFAHR: Mausefalle und Spinne! - DER MANN, DER IMMER KLEINER WURDE Clip (2025) Exklusiv
0l16I15awJ4 -> DIE BESTEN NEWS im November: Jurassic World 5, Die Mumie 4, Jumanji 3, Zelda Film... KinoCheck News
0tL-Bj3l37Q -> THE WITCHER Staffel 4 Neuer Trailer German Deutsch (2025) Netflix
2JZXCoR3bPc -> AVATAR 3: Fire and Ash Trailer 2 German Deutsch (2025)


In [16]:
# Einträge normalisieren damit Titel mit Movie Success übereinstimmen
import re

REMOVE_WORDS = [
    "trailer",
    "german",
    "deutsch",
    "clip",
    "exklusiv",
    "&",
]

def normalize_title(title: str):
    """
    Gibt (normalized_title, skip_flag) zurück
    """
    if not title:
        return None, True

    t = title.lower()

    # Überspringen wenn es ein Zusammenschnitt oder Serientrailer ist
    if t.startswith("die besten"):
        return None, True
    if "staffel" in t:
        return None, True

    # Jahreszahlen in Klammern entfernen
    t = re.sub(r"\(\s*\d{4}\s*\)", "", t)

    # Unerwünschte Wörter entfernen (als ganze Wörter)
    for w in REMOVE_WORDS:
        t = re.sub(rf"\b{re.escape(w)}\b", "", t)

    # Sonderzeichen & mehrfachen Whitespace aufräumen
    t = re.sub(r"\s+", " ", t).strip()

    # Leere Titel abfangen
    if not t:
        return None, True

    return t, False


In [17]:
# Testen ob Normalisieren klappt
examples = [
    "MOTHER'S BABY Trailer German Deutsch (2025) Exklusiv",
    "Hexenverbrennung ohne Gnade! - DER MEDICUS 2 Clip & Trailer German Deutsch (2025)",
    "Die besten Horrorfilme 2024",
    "Breaking Bad Staffel 2 Trailer"
]

for e in examples:
    print(e, "→", normalize_title(e))


MOTHER'S BABY Trailer German Deutsch (2025) Exklusiv → ("mother's baby", False)
Hexenverbrennung ohne Gnade! - DER MEDICUS 2 Clip & Trailer German Deutsch (2025) → ('hexenverbrennung ohne gnade! - der medicus 2 &', False)
Die besten Horrorfilme 2024 → (None, True)
Breaking Bad Staffel 2 Trailer → (None, True)


In [22]:
# Videotitel in hbase normalisieren
table = conn.table("kinocheck_videos")

updated = 0
skipped = 0

for rk, data in table.scan():
    raw_title = data.get(b"yt:title")
    if not raw_title:
        continue

    raw_title = raw_title.decode("utf-8", errors="ignore")
    clean_title, skip = normalize_title(raw_title)

    updates = {}

    if skip:
        updates[b"yt:skip"] = b"true"
        skipped += 1
    else:
        updates[b"yt:title_clean"] = clean_title.encode("utf-8")
        updates[b"yt:skip"] = b"false"
        updated += 1

    table.put(rk, updates)

print("Bereinigte Titel:", updated)
print("Übersprungene Videos:", skipped)


Bereinigte Titel: 221
Übersprungene Videos: 29


In [25]:
# Checken ob Normalisieren in hbase funktioniert hat

import happybase

HBASE_HOST = "hbase"
HBASE_PORT = 9090

def scan_sample(table_name: str, limit: int = 10):
    conn = happybase.Connection(host=HBASE_HOST, port=HBASE_PORT, timeout=20000)
    conn.open()
    try:
        table = conn.table(table_name)
        i = 0
        for rk, data in table.scan():
            title = data.get(b"yt:title", b"").decode("utf-8", errors="ignore")
            title_clean = data.get(b"yt:title_clean", b"").decode("utf-8", errors="ignore")
            skip = data.get(b"yt:skip", b"").decode("utf-8", errors="ignore")

            print(f"{i+1:02d}. {title}  |  {title_clean}  |  skip={skip}")
            i += 1
            if i >= limit:
                break
    finally:
        conn.close()

# --- Check: 10 Beispiele anzeigen ---
scan_sample("kinocheck_videos", limit=10)


01. Hexenverbrennung ohne Gnade! - DER MEDICUS 2 Clip & Trailer German Deutsch (2025)  |  hexenverbrennung ohne gnade! - der medicus 2 &  |  skip=false
02. MOTHER'S BABY Trailer German Deutsch (2025) Exklusiv  |  mother's baby  |  skip=false
03. Die besten COMEDY Filme 2025 & 2026 (Trailer German Deutsch)  |    |  skip=true
04. DER ASTRONAUT Trailer 2 German Deutsch (2026) Ryan Gosling, Sandra Hüller  |  der astronaut 2 ryan gosling, sandra hüller  |  skip=false
05. SPRINGSTEEN: Deliver Me From Nowhere Trailer 2 German Deutsch (2025)  |  springsteen: deliver me from nowhere 2  |  skip=false
06. Martin Freeman ist der Hüter der Zeit! - MOMO Clip & Trailer German Deutsch (2025)  |  martin freeman ist der hüter der zeit! - momo &  |  skip=false
07. LEBENSGEFAHR: Mausefalle und Spinne! - DER MANN, DER IMMER KLEINER WURDE Clip (2025) Exklusiv  |  lebensgefahr: mausefalle und spinne! - der mann, der immer kleiner wurde  |  skip=false
08. DIE BESTEN NEWS im November: Jurassic World 5, Die Mum